# Tensor Ops Illustrated

This notebook teaches tensor operations one by one. Obvious elementwise families are grouped because they share the same mental model. Everything else gets its own teaching section.

Use the loop:

```txt
intuition -> predict shape/value -> run -> read output -> connect to ML code
```


In [ ]:
import { tensor, empty, zeros, ones, full, rand, randn, seed, arange, linspace, cast, move, contiguous, finite_summary, finite_abs_max, add, sub, mul, div, neg, abs, sign, exp, log, sqrt, square, clamp, relu, sigmoid, tanh, gelu, silu, swish, softmax, dot, matmul, sum, mean, max, min, variance, std, argmax, argmin, where, masked_fill, gt_scalar, reshape, transpose, permute, squeeze, unsqueeze, cat, stack, at, slice, range, gather, index_select, topk, one_hot } from 'affon:compute'

function show(label: string, value: any) {
  console.log(label)
  if (typeof value === 'number') {
    console.log(value)
    return
  }
  console.log('shape:', value.shape)
  console.log('dtype:', value.dtype)
  console.log('device:', value.device)
  console.log(JSON.stringify(value.to_array()))
}


## tensor

**What it solves:** turns JavaScript numbers or nested arrays into Affon tensors.

**Shape rule:** nesting depth becomes rank; lengths become axis sizes.

**Before you run:** predict the shape of `[[1, 2], [3, 4]]`.


In [ ]:
{
const x = tensor([[1, 2], [3, 4]])
show('tensor([[1, 2], [3, 4]])', x)
console.log('Read it: two rows, two columns -> shape [2, 2].')
}


## empty

**What it solves:** allocates tensor storage when you plan to fill it later.

**Debug cue:** do not teach or inspect `empty` values as meaningful data; only its shape/dtype/device are meaningful before writing.


In [ ]:
{
const scratch = empty([2, 3])
console.log('empty([2, 3]) metadata:', scratch.shape, scratch.dtype, scratch.device)
}


## zeros, ones, full

**What they solve:** create tensors from a shape and a simple fill rule.

**Before you run:** predict the value in every cell of each output.


In [ ]:
{
show('zeros([2, 3])', zeros([2, 3]))
show('ones([2, 3])', ones([2, 3]))
show('full([2, 2], 7)', full([2, 2], 7))
}


## rand, randn, seed

**What they solve:** create random tensors. `rand` samples uniform values; `randn` samples normal-like values. `seed` makes examples repeatable.

**Before you run:** expect the two `rand` calls after the same seed to match.


In [ ]:
{
seed(42)
show('rand([3]) after seed(42)', rand([3]))
seed(42)
show('rand([3]) again after seed(42)', rand([3]))
seed(42)
show('randn([3]) after seed(42)', randn([3]))
}


## arange

**What it solves:** creates a counted number line.

**Rule:** `arange(start, end, step)` includes values before `end`.


In [ ]:
{
show('arange(0, 7, 2)', arange(0, 7, 2))
}


## linspace

**What it solves:** creates evenly spaced points across an interval.

**Before you run:** with 5 points from 0 to 1, predict the middle value.


In [ ]:
{
show('linspace(0, 1, 5)', linspace(0, 1, 5))
}


## cast

**What it solves:** changes dtype while preserving logical values.

**ML connection:** cast when a kernel or downstream operation expects a different numeric type.


In [ ]:
{
const x = tensor([1, 2, 3])
show('x', x)
show('cast(x, f64)', cast(x, 'f64'))
}


## move

**What it solves:** changes device placement.

**Debug cue:** values may look identical while device metadata changes.


In [ ]:
{
const x = tensor([1, 2, 3])
show('move(x, cpu)', move(x, 'cpu'))
}


## contiguous

**What it solves:** materializes a tensor into contiguous layout.

**ML connection:** layout matters for kernels and performance, even when values are the same.


In [ ]:
{
const x = transpose(tensor([[1, 2, 3], [4, 5, 6]]), 0, 1)
show('transposed x', x)
show('contiguous(x)', contiguous(x))
}


## finite_summary, finite_abs_max

**What they solve:** quick debugging checks for bad numeric values and large magnitudes.

**Before you run:** predict whether `[1, Infinity, NaN]` is finite-ok.


In [ ]:
{
console.log('finite_summary:', JSON.stringify(finite_summary(tensor([1, Infinity, NaN]))))
console.log('finite_abs_max:', finite_abs_max(tensor([-3, 2])))
}


## add, sub, mul, div

**What they solve:** combine matching positions. These are grouped because they share the same elementwise shape rule.

**Shape rule:** input shapes must be compatible; output follows the elementwise result shape.

**Before you run:** compute the top-left result for each op by hand.


In [ ]:
{
const a = tensor([[-2, -1, 0], [1, 2, 3]])
const b = tensor([[10, 10, 10], [20, 20, 20]])
show('add(a, b)', add(a, b))
show('sub(b, a)', sub(b, a))
show('mul(a, b)', mul(a, b))
show('div(b, tensor([[2,5,10],[4,5,10]]))', div(b, tensor([[2, 5, 10], [4, 5, 10]])))
}


## neg, abs, sign, square

**What they solve:** transform each value independently. These are grouped because every output value depends on exactly one input value.


In [ ]:
{
const x = tensor([-3, -1, 0, 2])
show('neg(x)', neg(x))
show('abs(x)', abs(x))
show('sign(x)', sign(x))
show('square(x)', square(x))
}


## exp

**What it solves:** exponentiates values, often before normalization.

**ML connection:** softmax uses exponentials internally.


In [ ]:
{
show('exp([-1, 0, 1])', exp(tensor([-1, 0, 1])))
}


## log

**What it solves:** maps multiplicative scale into additive scale; common in losses and probabilities.

**Debug cue:** log needs positive inputs.


In [ ]:
{
show('log([1, 2, 4])', log(tensor([1, 2, 4])))
}


## sqrt

**What it solves:** square root, often used after variance to get standard deviation.

**Debug cue:** sqrt needs non-negative inputs.


In [ ]:
{
show('sqrt([[1, 4, 9], [16, 25, 36]])', sqrt(tensor([[1, 4, 9], [16, 25, 36]])))
}


## clamp

**What it solves:** clips values into a range.

**Before you run:** decide what happens to values below -1 and above 1.


In [ ]:
{
show('clamp([-3, -1, 0, 2], -1, 1)', clamp(tensor([-3, -1, 0, 2]), -1, 1))
}


## relu

**What it solves:** keeps positive signal and zeros negative signal.

**Formula:** `max(x, 0)`.


In [ ]:
{
show('relu([-2, -1, 0, 1, 2])', relu(tensor([-2, -1, 0, 1, 2])))
}


## sigmoid

**What it solves:** squeezes values into `(0, 1)`, useful for probability-like binary scores.


In [ ]:
{
show('sigmoid([-4, -1, 0, 1, 4])', sigmoid(tensor([-4, -1, 0, 1, 4])))
}


## tanh

**What it solves:** squeezes values into `(-1, 1)` while staying centered at zero.


In [ ]:
{
show('tanh([-4, -1, 0, 1, 4])', tanh(tensor([-4, -1, 0, 1, 4])))
}


## gelu

**What it solves:** smooth ReLU-like gating, common in transformer feed-forward blocks.


In [ ]:
{
show('gelu([-4, -1, 0, 1, 4])', gelu(tensor([-4, -1, 0, 1, 4])))
}


## silu, swish

**What they solve:** smooth gating using `x * sigmoid(x)`. They are grouped because Affon exposes both names for the same teaching idea.


In [ ]:
{
const x = tensor([-4, -1, 0, 1, 4])
show('silu(x)', silu(x))
show('swish(x)', swish(x))
}


## dot

**What it solves:** multiply pairs and sum them into a scalar.

**Formula:** `dot(a,b)=sum_i a_i*b_i`.


In [ ]:
{
show('dot([1,2,3], [4,5,6])', dot(tensor([1, 2, 3]), tensor([4, 5, 6])))
}


## matmul

**What it solves:** row-by-column multiplication.

**High-rank rule:** leading dimensions are parallel.

**Before you run:** for `[2,3,4] @ [4,2]`, predict `[2,3,2]`.


In [ ]:
{
const x = tensor([[[1, 2, 3, 4], [10, 20, 30, 40], [100, 200, 300, 400]], [[5, 6, 7, 8], [50, 60, 70, 80], [500, 600, 700, 800]]])
const w = tensor([[1, 0], [0, 1], [1, 1], [2, -1]])
show('matmul(x, w)', matmul(x, w))
}


## sum

**What it solves:** reduces selected values by adding them.

**Before you run:** which axis disappears in `sum(x, 2)`?


In [ ]:
{
const x = tensor([[[1, 2, 3], [4, 5, 6]], [[10, 20, 30], [40, 50, 60]]])
show('sum(x, 2)', sum(x, 2))
}


## mean

**What it solves:** reduces selected values by averaging them.

**Before you run:** compare `keep=false` and `keep=true`.


In [ ]:
{
const x = tensor([[[1, 2, 3], [4, 5, 6]], [[10, 20, 30], [40, 50, 60]]])
show('mean(x, 2)', mean(x, 2))
show('mean(x, 2, true)', mean(x, 2, true))
}


## max

**What it solves:** returns the largest value along a selected axis.


In [ ]:
{
const x = tensor([[[1, 2, 3], [4, 5, 6]], [[10, 20, 30], [40, 50, 60]]])
show('max(x, 2)', max(x, 2))
}


## min

**What it solves:** returns the smallest value along a selected axis.


In [ ]:
{
const x = tensor([[[1, 2, 3], [4, 5, 6]], [[10, 20, 30], [40, 50, 60]]])
show('min(x, 2)', min(x, 2))
}


## variance

**What it solves:** measures average squared spread around the mean.


In [ ]:
{
const x = tensor([[[1, 2, 3], [4, 5, 6]], [[10, 20, 30], [40, 50, 60]]])
show('variance(x, 2)', variance(x, 2))
}


## std

**What it solves:** square root of variance, so spread is in the original units.


In [ ]:
{
const x = tensor([[[1, 2, 3], [4, 5, 6]], [[10, 20, 30], [40, 50, 60]]])
show('std(x, 2)', std(x, 2))
}


## argmax

**What it solves:** returns the index of the largest value, not the value itself.


In [ ]:
{
const x = tensor([[1, 5, 3], [9, 2, 7]])
show('argmax(x, 1)', argmax(x, 1))
}


## argmin

**What it solves:** returns the index of the smallest value, not the value itself.


In [ ]:
{
const x = tensor([[1, 5, 3], [9, 2, 7]])
show('argmin(x, 1)', argmin(x, 1))
}


## softmax

**What it solves:** normalizes one axis into distribution-like values that sum to about 1.

**Before you run:** does the softmax axis disappear? It does not; shape is preserved.


In [ ]:
{
const scores = tensor([[[1, 2, 3], [2, 0, -1]]])
show('softmax(scores, 2)', softmax(scores, 2))
}


## gt_scalar

**What it solves:** builds a mask by comparing every value to a scalar threshold.


In [ ]:
{
const x = tensor([[-2, -1, 0], [1, 2, 3]])
show('gt_scalar(x, 0)', gt_scalar(x, 0))
}


## where

**What it solves:** chooses value from `a` or `b` at each position using a mask.


In [ ]:
{
const x = tensor([[-2, -1, 0], [1, 2, 3]])
const mask = gt_scalar(x, 0)
show('where(mask, full([2,3], 99), x)', where(mask, full([2, 3], 99), x))
}


## masked_fill

**What it solves:** replaces masked positions with one scalar value.


In [ ]:
{
const x = tensor([[-2, -1, 0], [1, 2, 3]])
show('masked_fill(x, gt_scalar(x, 0), 99)', masked_fill(x, gt_scalar(x, 0), 99))
}


## reshape

**What it solves:** gives the same values a new axis grouping.

**Debug cue:** total element count must stay the same.


In [ ]:
{
show('reshape([[1,2,3],[4,5,6]], [3,2])', reshape(tensor([[1, 2, 3], [4, 5, 6]]), [3, 2]))
}


## transpose

**What it solves:** swaps two axes.


In [ ]:
{
show('transpose([[1,2,3],[4,5,6]], 0, 1)', transpose(tensor([[1, 2, 3], [4, 5, 6]]), 0, 1))
}


## permute

**What it solves:** reorders all axes using an explicit axis order.


In [ ]:
{
const x = tensor([[[1, 2], [3, 4], [5, 6]]])
show('permute(x, [1,0,2])', permute(x, [1, 0, 2]))
}


## squeeze

**What it solves:** removes size-1 axes.


In [ ]:
{
const x = tensor([[[1, 2, 3], [4, 5, 6]]])
show('squeeze(x, 0)', squeeze(x, 0))
}


## unsqueeze

**What it solves:** inserts a size-1 axis.


In [ ]:
{
const x = tensor([[1, 2, 3], [4, 5, 6]])
show('unsqueeze(x, 0)', unsqueeze(x, 0))
}


## cat

**What it solves:** joins tensors along an existing axis.

**Before you run:** predict why `dim=0` grows rows and `dim=1` grows columns.


In [ ]:
{
const x = tensor([[1, 2, 3], [4, 5, 6]])
show('cat([x,x], 0)', cat([x, x], 0))
show('cat([x,x], 1)', cat([x, x], 1))
}


## stack

**What it solves:** creates a new axis to hold multiple tensors.


In [ ]:
{
const x = tensor([[1, 2, 3], [4, 5, 6]])
show('stack([x,x], 0)', stack([x, x], 0))
}


## at

**What it solves:** selects one explicit position.


In [ ]:
{
const x = tensor([[1, 2, 3], [4, 5, 6]])
show('at(x, 1, 2)', at(x, 1, 2))
}


## slice and range

**What they solve:** select ranges or fixed positions. `range` describes a span for `slice`.


In [ ]:
{
const x = tensor([[1, 2, 3], [4, 5, 6]])
show('slice(x, range(0, 2), 1)', slice(x, range(0, 2), 1))
}


## gather

**What it solves:** selects per-position indices from an input tensor.

**Debug cue:** use `gather` when each row/position has its own indices.


In [ ]:
{
const x = tensor([[1, 2, 3], [4, 5, 6]])
const index = tensor([[2, 1], [0, 2]], { dtype: 'i64' })
show('gather(x, 1, index)', gather(x, 1, index))
}


## index_select

**What it solves:** selects the same list of indices along one axis.


In [ ]:
{
const x = tensor([[1, 2, 3], [4, 5, 6]])
show('index_select(x, 0, [1,0])', index_select(x, 0, tensor([1, 0], { dtype: 'i64' })))
}


## topk

**What it solves:** returns the largest `k` values and their indices along an axis.


In [ ]:
{
const x = tensor([[1, 2, 3], [4, 5, 6]])
const tk = topk(x, 2, 1)
show('topk values', tk.values)
show('topk indices', tk.indices)
}


## one_hot

**What it solves:** turns integer class ids into indicator vectors.

**ML connection:** useful for class targets or token ids when a dense indicator representation is needed.


In [ ]:
{
show('one_hot([0,2,1], 3)', one_hot(tensor([0, 2, 1], { dtype: 'i64' }), 3))
}


## Reading Tensor Ops In Practice

For any unfamiliar tensor op, ask these questions:

1. Does it work element by element?
2. Does it reduce an axis?
3. Does it create a new axis?
4. Does it reorder or select existing values?
5. Which axes are just parallel context?

That habit is more important than memorizing every formula.
